# 11 — Packaging & Tooling
**References:** Python Packaging User Guide · `uv` docs · PEP 517, 518, 621, 660

## Narrative thread
```
pyproject.toml -> uv workflow -> Build backends -> Publishing to PyPI -> CI/CD -> Pre-commit
```

## 1. `pyproject.toml` — the unified config file

Since PEP 517/518 (2018), `pyproject.toml` is the standard for Python project metadata. It replaces `setup.py`, `setup.cfg`, `requirements.txt`, `.flake8`, `mypy.ini`, etc.

```toml
[build-system]
requires      = ["hatchling"]       # or "setuptools", "flit-core", "pdm-backend"
build-backend = "hatchling.build"

[project]                           # PEP 621: standard metadata
name        = "my-package"
version     = "0.1.0"
description = "A short description"
readme      = "README.md"
license     = { text = "MIT" }
requires-python = ">=3.11"
dependencies = [
    "numpy>=1.26",
    "httpx>=0.27",
]

[project.optional-dependencies]
dev  = ["pytest>=8", "mypy", "ruff"]
docs = ["mkdocs-material"]

[project.scripts]
my-cli = "my_package.cli:main"      # entry point

[tool.mypy]
strict = true
python_version = "3.11"

[tool.ruff]
line-length = 100
select = ["E", "F", "I", "UP", "N"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts   = "-v --tb=short"
```

## 2. `uv` — the modern Python toolchain

`uv` (by Astral) replaces `pip`, `pip-tools`, `virtualenv`, `pyenv`, `pipx` and `poetry` in most workflows. Written in Rust — 10–100× faster than pip.

### Essential `uv` commands

```bash
# Project management
uv init my-project              # create new project with pyproject.toml
uv sync                         # create venv + install all deps from uv.lock
uv add numpy pandas             # add runtime dependencies
uv add --dev pytest mypy ruff   # add dev dependencies
uv remove httpx                 # remove a dependency

# Running code
uv run python script.py         # run in the project venv
uv run pytest                   # run tests
uv run mypy .                   # type check

# Python version management
uv python install 3.12          # download and install Python 3.12
uv python pin 3.11              # pin project to Python 3.11

# Tools (isolated, like pipx)
uv tool install ruff             # install ruff as a global tool
uv tool run black .              # run black without installing

# Building and publishing
uv build                        # create .tar.gz and .whl in dist/
uv publish                      # publish to PyPI (or --repository testpypi)

# Lock file
uv lock                         # regenerate uv.lock without installing
uv export --format requirements-txt > requirements.txt  # compatibility
```

### `uv.lock` vs `requirements.txt`

| | `uv.lock` | `requirements.txt` |
|---|---|---|
| Format | TOML — human-readable, includes hashes | Plain text |
| Resolution | Cross-platform (OS/Python matrix) | Single-platform |
| Precision | Exact versions + hashes for all deps | Often unpinned |
| Commit? | **Yes** — reproducible builds | Sometimes |
| Used by | `uv sync` | `pip install -r` |

In [ ]:
import subprocess, sys

# ── Show what uv knows about the current environment ─────────────────────
print('=== uv environment info ===')
result = subprocess.run(['uv', 'python', 'find'], capture_output=True, text=True)
print(f'Python: {result.stdout.strip()}')

result = subprocess.run(['uv', 'pip', 'list', '--format=columns'],
                        capture_output=True, text=True)
if result.returncode == 0:
    lines = result.stdout.strip().split('\n')
    print(f'\nInstalled packages ({len(lines)-2} packages):')
    for line in lines[:8]:
        print(f'  {line}')
    if len(lines) > 8:
        print(f'  ... and {len(lines)-8} more')
else:
    print(f'uv pip list error: {result.stderr[:200]}')

## 3. Code quality tools

### `ruff` — the all-in-one linter and formatter

`ruff` (Rust, Astral) replaces `flake8`, `isort`, `pyupgrade`, `bandit`, and optionally `black`.

```bash
uv run ruff check .          # lint
uv run ruff check --fix .    # auto-fix
uv run ruff format .         # format (black-compatible)
```

### `mypy` — type checker

```bash
uv run mypy src/ --strict
```

### Pre-commit hooks

```yaml
# .pre-commit-config.yaml
repos:
  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.4.0
    hooks:
      - id: ruff
        args: [--fix]
      - id: ruff-format

  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.9.0
    hooks:
      - id: mypy
        args: [--strict]
```

## 4. CI with GitHub Actions

```yaml
# .github/workflows/ci.yml
name: CI
on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ['3.11', '3.12', '3.13']
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v2
        with:
          python-version: ${{ matrix.python-version }}
      - run: uv sync --all-extras
      - run: uv run pytest --cov=src --cov-report=xml
      - run: uv run mypy src --strict
      - run: uv run ruff check .
      - uses: codecov/codecov-action@v4
```

## 5. Key takeaways

| Tool | Replaces |
|---|---|
| `uv` | `pip`, `virtualenv`, `pyenv`, `pip-tools`, `pipx` |
| `ruff` | `flake8`, `isort`, `pyupgrade`, optionally `black` |
| `mypy --strict` | No untyped code; catches most type errors |
| `pyproject.toml` | `setup.py`, `setup.cfg`, `requirements.txt`, tool configs |
| `uv.lock` | `requirements.txt` — commit it; reproducible installs |

**Next:** notebook 12 — CPython internals: bytecode, frames, and the C API.